# Test de classification LLM pour l’identification des articles hors-échantillon

Ce notebook constitue une étape préparatoire à la classification finale du corpus TASS par LLM. Son objectif est de tester différents modèles sur un jeu d’articles déjà annotés, afin d’évaluer leur capacité à distinguer les articles pertinents pour l’étude de ceux qui doivent être exclus.

La tâche est formulée comme une classification binaire autour d’un seul label : **hors-échantillon**. Dans ce cadre, `hors_echantillon = false` signifie que l’article est conservé dans le corpus d’analyse, car il traite de la migration, de l’évacuation ou de l’accueil de personnes venues d’Ukraine, du Donbass ou des territoires sous contrôle russe vers la Russie, la Crimée ou l’Ossétie du Sud. À l’inverse, `hors_echantillon = true` signifie que l’article ne porte pas sur ce phénomène et doit être écarté.

Le notebook suit quatre étapes principales : préparation du jeu de référence annoté, construction du prompt d’annotation, lancement des prédictions LLM sur le test set, puis évaluation des résultats à partir des métriques de classification.

## 1. Préparation du jeu de référence annoté

Cette première partie rassemble les annotations humaines utilisées comme référence pour l’évaluation. Les deux fichiers de réponses sont chargés, concaténés, puis renumérotés afin d’obtenir un test set unique avec des identifiants continus.

In [7]:
import json 

In [48]:
with open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/llm_test_answers_full.json", "r") as f:
    data_100 = json.load(f)

with open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/llm_test_answers_next_100.json", "r") as f:
    data_200 = json.load(f)

data_100

[{'text': 'МОСКВА, 18 февраля. /ТАСС/. Уполномоченный при президенте РФ по правам ребенка Мария Львова-Белова готова оказать необходимую помощь эвакуирующимся с территории Донецкой и Луганской народных республик (ДНР и ЛНР) детям и семьям. Об этом в пятницу сообщила пресс-служба омбудсмена.\n"В настоящий момент отсутствуют подробности о количестве эвакуируемых и местах их размещения. Институт уполномоченных по правам ребенка готов оказать все необходимое содействие в помощи прибывающим с территории ЛНР и ДНР детям и семьям с детьми", - сказано в сообщении.\nТакже сообщается, что Львова-Белова находится на связи с детским омбудсменом Ростовской области.\nДонецкая и Луганская народные республики на фоне серьезного обострения ситуации в Донбассе объявили в пятницу об организации выезда населения в Россию. Губернатор Ростовской области Василий Голубев запросил помощь федерального центра. Как сообщил первый зампредседателя комитета Госдумы по делам СНГ, евразийской интеграции и связям с соо

In [6]:
data_200

[{'text': 'ЛУГАНСК, 22 ноября. /ТАСС/. Более 110 жителей городов Лисичанск, Рубежное и Северодонецк, входящих в Северодонецкую агломерацию, временно размещены в социальных учреждениях, подведомственных Министерству труда и социальной политики Луганской Народной Республики (ЛНР). Об этом сообщила во вторник пресс-служба правительства региона.\n"Социальные учреждения, подведомственные министерству труда и социальной политики, с 30 октября по 17 ноября приняли из Северодонецка, Лисичанска и Рубежного 111 человек, нуждавшихся во временном размещении", - говорится в тексте, опубликованном в Telegram-канале правительства.\nПо данным властей, в санаториях государственного учреждения ЛНР "Республиканская здравница" размещены 62 человека - это семьи с детьми, инвалиды и пожилые люди. "В интернатных учреждениях - 49 человек, в том числе 46 человек - в антрацитовском доме-интернате, трое - в луганском гериатрическом пансионате №1", - отмечается в сообщении.\nРанее врио главы республики Леонид Пас

In [49]:
HORS_LABEL = "Article hors échantillon - ne parle pas de la migration de l'Ukraine vers la Russie"

combined = data_100 + data_200

for new_id, item in enumerate(combined, start=1):
    item["id"] = new_id

print(len(combined))
print(combined[0]["id"], combined[-1]["id"])

200
1 200


### 1.1. Réduction de l’annotation au label hors-échantillon

Les annotations initiales peuvent contenir plusieurs catégories narratives. Ici, on ne conserve qu’une information : l’article appartient-il ou non à la catégorie **hors-échantillon**.

Si le label « Article hors échantillon » est présent, l’article est marqué comme hors-échantillon. Dans le cas contraire, il est considéré comme pertinent pour le corpus d’analyse.

In [50]:
for item in combined:
    labels = item.get("narrative", [])

    if HORS_LABEL in labels:
        item["narrative"] = [HORS_LABEL]
    else:
        item["narrative"] = []

    item.pop("narratives", None)

output_path = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/llm_test_answers_200_hors_only.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print(len(combined))
print(combined[:2])

200
[{'text': 'МОСКВА, 18 февраля. /ТАСС/. Уполномоченный при президенте РФ по правам ребенка Мария Львова-Белова готова оказать необходимую помощь эвакуирующимся с территории Донецкой и Луганской народных республик (ДНР и ЛНР) детям и семьям. Об этом в пятницу сообщила пресс-служба омбудсмена.\n"В настоящий момент отсутствуют подробности о количестве эвакуируемых и местах их размещения. Институт уполномоченных по правам ребенка готов оказать все необходимое содействие в помощи прибывающим с территории ЛНР и ДНР детям и семьям с детьми", - сказано в сообщении.\nТакже сообщается, что Львова-Белова находится на связи с детским омбудсменом Ростовской области.\nДонецкая и Луганская народные республики на фоне серьезного обострения ситуации в Донбассе объявили в пятницу об организации выезда населения в Россию. Губернатор Ростовской области Василий Голубев запросил помощь федерального центра. Как сообщил первый зампредседателя комитета Госдумы по делам СНГ, евразийской интеграции и связям с

In [51]:
#check unique labels in 'narrative' 

unique_labels = set()
for item in combined:
    labels = item.get("narrative", [])
    unique_labels.update(labels)    
print(unique_labels)

{"Article hors échantillon - ne parle pas de la migration de l'Ukraine vers la Russie"}


### 1.2. Transformation en variable booléenne

Pour faciliter l’évaluation automatique, le label textuel est transformé en variable booléenne : `true` pour les articles hors-échantillon et `false` pour les articles à conserver. Le fichier obtenu constitue la vérité de terrain utilisée pour comparer les prédictions des modèles.

In [52]:
#hors echantillon = true/false 

for item in combined:
    labels = item.get("narrative", [])
    item["hors_echantillon"] = HORS_LABEL in labels
    item.pop("narrative", None)

print(combined[:2])

[{'text': 'МОСКВА, 18 февраля. /ТАСС/. Уполномоченный при президенте РФ по правам ребенка Мария Львова-Белова готова оказать необходимую помощь эвакуирующимся с территории Донецкой и Луганской народных республик (ДНР и ЛНР) детям и семьям. Об этом в пятницу сообщила пресс-служба омбудсмена.\n"В настоящий момент отсутствуют подробности о количестве эвакуируемых и местах их размещения. Институт уполномоченных по правам ребенка готов оказать все необходимое содействие в помощи прибывающим с территории ЛНР и ДНР детям и семьям с детьми", - сказано в сообщении.\nТакже сообщается, что Львова-Белова находится на связи с детским омбудсменом Ростовской области.\nДонецкая и Луганская народные республики на фоне серьезного обострения ситуации в Донбассе объявили в пятницу об организации выезда населения в Россию. Губернатор Ростовской области Василий Голубев запросил помощь федерального центра. Как сообщил первый зампредседателя комитета Госдумы по делам СНГ, евразийской интеграции и связям с соо

In [53]:
with open ("/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/hors_echantillon_answers.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

## 2. Définition du prompt de classification

Cette section définit les consignes données au modèle. Le prompt décrit précisément les cas qui doivent être classés comme `hors_echantillon = false` et ceux qui doivent être classés comme `hors_echantillon = true`.

Une attention particulière est portée aux cas ambigus : évacuations vers la Russie, accueil en Crimée, enfants déplacés, aide humanitaire, PVR, couloirs humanitaires ou dispositifs administratifs. L’objectif est de réduire l’instabilité des réponses du modèle en explicitant les frontières du corpus.

In [ ]:
SYSTEM_PROMPT = """
Tu es un spécialiste de l’annotation de discours médiatiques.

Ta tâche est uniquement de déterminer si un article TASS en russe doit recevoir le label suivant :

"Article hors échantillon - ne parle pas de la migration de l'Ukraine vers la Russie"

========================
HORS_ECHANTILLON = FALSE 
========================

Choisis hors_echantillon = false si l’article traite principalement de la migration des gens provenants de l'Ukraine, Donetsk, Louhansk vers la Russie, notamment en ce qui concerne :

- le déplacement vers la Russie ;
- l’arrivée en Russie ;
- les évacuations vers la Russie ;
- l'ouverture des couloirs humanitaires par la Russie en Ukraine pour évacuer des civils vers la Russie ;
- l’accueil en Russie ;
- l’installation en Russie ;
- l’hébergement en Russie ;
- la prise en charge en Russie ;
- les conditions de vie de ces gens d'Ukraine en Russie ;
- les PVR, centres d’accueil, points d’hébergement ou structures de placement pour ces gens en Russie ;
- l’aide administrative, médicale, scolaire, sociale ou financière pour ces gens en Russie ;
- les lois, décisions, des dispositifs, les mesures sociales, juridiques, financières ou administratives destinées aux évacués, réfugiés, déplacés ou migrants venus d’Ukraine / Donbass, même si l’article est présenté comme une réunion politique ou institutionnelle ;
- l’envoi d’aide humanitaire vers la Russie, Rostov, Belgorod, la Crimée ou d’autres régions russes pour des réfugiés, déplacés ou évacués ;
- les enfants venant d’Ukraine, du Donbass, de la DNR, de la LNR, de Kherson, de Zaporijjia ou de régions sous contrôle russe accueillis en Russie, en Crimée ou en Ossétie du Sud.

IMPORTANT :
- Une évacuation vers la Russie = toujours false.
- Une évacuation vers la Crimée ou l’Ossétie du Sud = false.
- L’accueil en Russie d’enfants, familles ou civils venus d’Ukraine / Donbass = false.
- Tout article sur des PVR en Russie pour ces personnes = false.
- Tout article sur des enfants venus d’Ukraine / Donbass présents en Russie, en Crimée ou en Ossétie du Sud = false.
- Les couloirs humanitaires ouverts par la Russie en Ukraine pour évacuer des civils = false, même si l’évacuation commence en Ukraine.

Exemples false :
- « Вторая партия гуманитарного груза для эвакуированных из Донецкой и Луганской народных республик отправлена в Ростов-на-Дону. »
- « Крым готов принять порядка 5 тыс. детей из Херсонской области. »
- « Российские детские лагеря отдыха приняли детей из Донбасса и с освобожденных территорий Украины. »
- « Россия открыла гуманитарный коридор для эвакуации мирных жителей. »
- « Беженцам, прибывающим в российские регионы, выплачивают по 10 тыс. рублей. »

========================
HORS_ECHANTILLON = TRUE
========================

Réponds hors_echantillon = true si l’article porte principalement sur :

- l’accueil uniquement en Europe, en Pologne, en Allemagne ou dans un autre pays tiers, sans Russie / Crimée / Ossétie du Sud ;
- une aide humanitaire envoyée aux habitants du Donbass, aux habitants de la DNR/LNR, de Kherson, de Zaporijjia ou d’autres territoires, sans mention d’évacués, réfugiés ou déplacés vers la Russie ;
- une aide humanitaire formulée comme aide aux “habitants du Donbass” / “habitants de la DNR et de la LNR” = true ;
- des déplacés internes en Ukraine ou dans les territoires occupés, sans arrivée en Russie ;
- des déplacés internes de Russie vers d’autres régions russes, sans lien avec l’Ukraine / Donbass ;
- des évacuations à l’intérieur de l’Ukraine ou vers des territoires non russes ;
- une évacuation de l’Ukraine vers une autre partie de l’Ukraine ;
- la mobilisation, les combats, les opérations militaires, les bombardements ou la situation politique, sans accueil ni prise en charge en Russie ;
- l’aide à des militaires ou à des participants de l’opération militaire, et non à des réfugiés / déplacés / évacués ;
- des catastrophes naturelles ou événements internes en Russie sans lien avec la migration depuis l’Ukraine / Donbass vers la Russie ;
- des dispositifs politiques, administratifs ou civiques liés aux passeports, à la citoyenneté, aux référendums, aux élections, aux bureaux de vote ou au statut des territoires. 

IMPORTANT :
- “Aide aux habitants du Donbass / DNR / LNR” = true.
- Évacuation Ukraine → Ukraine = true.
- Si l’article parle uniquement de pays voisins ou de l’Europe, répondre true.
- Si l’article parle de plusieurs pays et inclut la Russie parmi les destinations, répondre true

Exemples true :
- « Около 80 тонн гуманитарной помощи для жителей Донецкой и Луганской народных республик отправили из Дагестана. »
- « Фура с продуктами, собранными для жителей Донбасса, отправилась в Ростовскую область » si le texte présente l’aide comme destinée aux habitants du Donbass et non aux évacués en Russie.
- « Восемь зарубежных избирательных участков организуют в Якутии для проведения референдумов... »
- « Крым окажет содействие в оформлении российских паспортов для жителей освобожденных территорий Украины. »
- « Две автобусные колонны с эвакуированными жителями города Сумы приехали в Полтавскую область. »

========================
FORMAT
========================

Réponds uniquement en JSON valide :

{"id": ..., "hors_echantillon": true}

ou

{"id": ..., "hors_echantillon": false}
"""

USER_PROMPT = """
Lis l’article TASS ci-dessous et détermine uniquement s’il doit être classé comme hors échantillon.

Retourne uniquement un JSON valide.
"""

## 3. Préparation des articles à classer par le LLM

Cette partie prépare le fichier d’entrée envoyé au modèle. Les deux fichiers de questions sont chargés, fusionnés et renumérotés de la même manière que le fichier de référence annoté.

Chaque ligne du fichier JSONL contient un article à classer, avec son identifiant et son texte. Ce format permet ensuite de traiter les articles un par un et de sauvegarder les résultats progressivement.

In [54]:
import jsonlines

# 1. Загружаем jsonl
questions_100 = []
with jsonlines.open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/llm_test_questions.jsonl", "r") as reader:
    for obj in reader:
        questions_100.append(obj)

questions_200 = []
with jsonlines.open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/llm_test_questions_next_100.jsonl", "r") as reader:
    for obj in reader:
        questions_200.append(obj)

questions_combined = questions_100 + questions_200

for new_id, item in enumerate(questions_combined, start=1):
    item["id"] = new_id

print(len(questions_combined))
print(questions_combined[0]["id"], questions_combined[-1]["id"])
print(questions_combined[0])

200
1 200
{'text': 'МОСКВА, 18 февраля. /ТАСС/. Уполномоченный при президенте РФ по правам ребенка Мария Львова-Белова готова оказать необходимую помощь эвакуирующимся с территории Донецкой и Луганской народных республик (ДНР и ЛНР) детям и семьям. Об этом в пятницу сообщила пресс-служба омбудсмена.\n"В настоящий момент отсутствуют подробности о количестве эвакуируемых и местах их размещения. Институт уполномоченных по правам ребенка готов оказать все необходимое содействие в помощи прибывающим с территории ЛНР и ДНР детям и семьям с детьми", - сказано в сообщении.\nТакже сообщается, что Львова-Белова находится на связи с детским омбудсменом Ростовской области.\nДонецкая и Луганская народные республики на фоне серьезного обострения ситуации в Донбассе объявили в пятницу об организации выезда населения в Россию. Губернатор Ростовской области Василий Голубев запросил помощь федерального центра. Как сообщил первый зампредседателя комитета Госдумы по делам СНГ, евразийской интеграции и свя

In [56]:
with open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/200_questions.jsonl", "w", encoding="utf-8") as f:
    for item in questions_combined:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")



### 3.1. Définition des chemins d’entrée et de sortie

Cette cellule indique le fichier JSONL contenant les articles à classer et le fichier dans lequel seront enregistrées les prédictions du modèle testé. En changeant le chemin de sortie ou le nom du modèle, le même notebook peut être utilisé pour comparer plusieurs LLM.

In [42]:
INPUT_JSONL = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/200_questions.jsonl"
OUTPUT_JSONL = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/llm_minigpt_hors_echantillon_3.jsonl"

In [43]:
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    n_lines = sum(1 for line in f if line.strip())

print(n_lines)

200


## 4. Connexion à l’API du modèle

Cette section configure l’accès au modèle via OpenRouter. La clé API doit être renseignée localement avant l’exécution. Le notebook utilise une température nulle afin de limiter la variabilité des réponses et de rendre la comparaison entre modèles plus stable.

In [ ]:
import requests
from openai import OpenAI

API_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = ""  # <-- replace with your actual API key
client = OpenAI(api_key=API_KEY, base_url=API_URL) 

## 5. Lancement du test de classification LLM

Cette cellule envoie chaque article du test set au modèle avec le prompt défini plus haut. Le modèle doit répondre uniquement sous forme de JSON valide contenant l’identifiant de l’article et la valeur booléenne `hors_echantillon`.

Les résultats sont écrits au fur et à mesure dans un fichier JSONL. Cette sauvegarde progressive est utile en cas d’interruption : les prédictions déjà produites ne sont pas perdues.

In [45]:

results = []

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```json", "", text).strip()
        text = re.sub(r"^```", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    return json.loads(text)

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        article = json.loads(line)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"{USER_PROMPT}\n\n"
                    f"ID: {article['id']}\n\n"
                    f"{article['text']}"
                )
            }
        ]

        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "openai/gpt-5.4-mini",
                "messages": messages,
                "temperature": 0
            }
        )

        response.raise_for_status()

        content = response.json()["choices"][0]["message"]["content"]
        parsed = extract_json(content)

        result = {
            "id": article["id"],
            "hors_echantillon": parsed["hors_echantillon"]
        }

        results.append(result)

        with open(OUTPUT_JSONL, "a", encoding="utf-8") as out:
            out.write(json.dumps(result, ensure_ascii=False) + "\n")

        print(f"Processed: {article['id']} -> hors_echantillon={result['hors_echantillon']}")

Processed: 1 -> hors_echantillon=False
Processed: 2 -> hors_echantillon=False
Processed: 3 -> hors_echantillon=False
Processed: 4 -> hors_echantillon=False
Processed: 5 -> hors_echantillon=False
Processed: 6 -> hors_echantillon=False
Processed: 7 -> hors_echantillon=False
Processed: 8 -> hors_echantillon=False
Processed: 9 -> hors_echantillon=False
Processed: 10 -> hors_echantillon=False
Processed: 11 -> hors_echantillon=False
Processed: 12 -> hors_echantillon=False
Processed: 13 -> hors_echantillon=False
Processed: 14 -> hors_echantillon=False
Processed: 15 -> hors_echantillon=False
Processed: 16 -> hors_echantillon=False
Processed: 17 -> hors_echantillon=False
Processed: 18 -> hors_echantillon=False
Processed: 19 -> hors_echantillon=False
Processed: 20 -> hors_echantillon=False
Processed: 21 -> hors_echantillon=True
Processed: 22 -> hors_echantillon=True
Processed: 23 -> hors_echantillon=False
Processed: 24 -> hors_echantillon=False
Processed: 25 -> hors_echantillon=True
Processed: 2

## 6. Vérification et évaluation des résultats

La partie suivante recharge les prédictions produites par le modèle et les compare aux annotations de référence. L’objectif est de vérifier que les identifiants correspondent bien, puis de mesurer les performances du modèle sur la tâche binaire de détection des articles hors-échantillon.

In [46]:
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

In [47]:
results

[{'id': 1, 'hors_echantillon': False},
 {'id': 2, 'hors_echantillon': False},
 {'id': 3, 'hors_echantillon': False},
 {'id': 4, 'hors_echantillon': False},
 {'id': 5, 'hors_echantillon': False},
 {'id': 6, 'hors_echantillon': False},
 {'id': 7, 'hors_echantillon': False},
 {'id': 8, 'hors_echantillon': False},
 {'id': 9, 'hors_echantillon': False},
 {'id': 10, 'hors_echantillon': False},
 {'id': 11, 'hors_echantillon': False},
 {'id': 12, 'hors_echantillon': False},
 {'id': 13, 'hors_echantillon': False},
 {'id': 14, 'hors_echantillon': False},
 {'id': 15, 'hors_echantillon': False},
 {'id': 16, 'hors_echantillon': False},
 {'id': 17, 'hors_echantillon': False},
 {'id': 18, 'hors_echantillon': False},
 {'id': 19, 'hors_echantillon': False},
 {'id': 20, 'hors_echantillon': False},
 {'id': 21, 'hors_echantillon': True},
 {'id': 22, 'hors_echantillon': True},
 {'id': 23, 'hors_echantillon': False},
 {'id': 24, 'hors_echantillon': False},
 {'id': 25, 'hors_echantillon': True},
 {'id': 26, 

In [59]:
with open ("/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/hors_echantillon_answers.json", "r", encoding="utf-8") as f:
    combined = json.load(f)

In [60]:
combined

[{'text': 'МОСКВА, 18 февраля. /ТАСС/. Уполномоченный при президенте РФ по правам ребенка Мария Львова-Белова готова оказать необходимую помощь эвакуирующимся с территории Донецкой и Луганской народных республик (ДНР и ЛНР) детям и семьям. Об этом в пятницу сообщила пресс-служба омбудсмена.\n"В настоящий момент отсутствуют подробности о количестве эвакуируемых и местах их размещения. Институт уполномоченных по правам ребенка готов оказать все необходимое содействие в помощи прибывающим с территории ЛНР и ДНР детям и семьям с детьми", - сказано в сообщении.\nТакже сообщается, что Львова-Белова находится на связи с детским омбудсменом Ростовской области.\nДонецкая и Луганская народные республики на фоне серьезного обострения ситуации в Донбассе объявили в пятницу об организации выезда населения в Россию. Губернатор Ростовской области Василий Голубев запросил помощь федерального центра. Как сообщил первый зампредседателя комитета Госдумы по делам СНГ, евразийской интеграции и связям с соо

### 6.1. Contrôle de l’alignement des identifiants

Avant de calculer les métriques, cette fonction vérifie que les prédictions et les annotations de référence contiennent les mêmes identifiants, dans le même ordre et sans doublons. Ce contrôle évite de produire une évaluation faussée par un décalage entre les fichiers.

In [61]:
def check_id_matching(predicted, annotated):
    pred_ids = [item["id"] for item in predicted]
    ann_ids = [item["id"] for item in annotated]

    print("Predicted length:", len(pred_ids))
    print("Annotated length:", len(ann_ids))

    print("Predicted duplicates:", sorted([x for x in set(pred_ids) if pred_ids.count(x) > 1])[:20])
    print("Annotated duplicates:", sorted([x for x in set(ann_ids) if ann_ids.count(x) > 1])[:20])

    print("Predicted min/max:", min(pred_ids), max(pred_ids))
    print("Annotated min/max:", min(ann_ids), max(ann_ids))

    mismatches = []

    for i, (p_id, a_id) in enumerate(zip(pred_ids, ann_ids)):
        if p_id != a_id:
            mismatches.append({
                "index": i,
                "predicted_id": p_id,
                "annotated_id": a_id
            })

    print("Position mismatches:", len(mismatches))
    print("First mismatches:", mismatches[:10])

    missing_in_pred = sorted(set(ann_ids) - set(pred_ids))
    extra_in_pred = sorted(set(pred_ids) - set(ann_ids))

    print("Missing in predicted:", missing_in_pred[:20])
    print("Extra in predicted:", extra_in_pred[:20])

    return mismatches

In [62]:
mismatches = check_id_matching(results, combined)

Predicted length: 200
Annotated length: 200
Predicted duplicates: []
Annotated duplicates: []
Predicted min/max: 1 200
Annotated min/max: 1 200
Position mismatches: 0
First mismatches: []
Missing in predicted: []
Extra in predicted: []


### 6.2. Calcul de l’accuracy

Cette fonction compare, pour chaque identifiant, la prédiction du modèle avec l’annotation de référence. Elle renvoie l’accuracy globale ainsi qu’un tableau détaillant les articles correctement et incorrectement classés.

In [63]:
def evaluate_hors_by_id(predicted, annotated):
    pred_by_id = {
        item["id"]: item["hors_echantillon"]
        for item in predicted
    }

    true_by_id = {
        item["id"]: item["hors_echantillon"]
        for item in annotated
    }

    all_ids = sorted(set(pred_by_id) | set(true_by_id))

    rows = []
    correct = 0

    for item_id in all_ids:
        pred = pred_by_id.get(item_id)
        true = true_by_id.get(item_id)

        is_correct = pred == true

        if is_correct:
            correct += 1

        rows.append({
            "id": item_id,
            "correct": is_correct,
            "predicted": pred,
            "annotated": true
        })

    accuracy = correct / len(all_ids) if all_ids else 0

    return accuracy, rows

In [64]:
accuracy, rows = evaluate_hors_by_id(results, combined)

print("Accuracy:", round(accuracy, 4))
print("Correct:", sum(r["correct"] for r in rows))
print("Total:", len(rows))
print("Wrong:", len([r for r in rows if not r["correct"]]))

Accuracy: 0.975
Correct: 195
Total: 200
Wrong: 5


### 6.3. Métriques détaillées de classification

Cette section calcule les métriques classiques pour une classification binaire : précision, rappel, F1-score, spécificité et balanced accuracy. Ces indicateurs permettent de mieux comprendre le comportement du modèle, notamment s’il tend à exclure trop d’articles ou, au contraire, à conserver des articles hors sujet.

In [65]:
def evaluate_hors_metrics(predicted, annotated):
    accuracy, rows = evaluate_hors_by_id(predicted, annotated)

    tp = fp = tn = fn = 0

    for r in rows:
        pred = r["predicted"]
        true = r["annotated"]

        if pred is True and true is True:
            tp += 1
        elif pred is True and true is False:
            fp += 1
        elif pred is False and true is False:
            tn += 1
        elif pred is False and true is True:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    specificity = tn / (tn + fp) if (tn + fp) else 0
    balanced_accuracy = (recall + specificity) / 2

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "total": len(rows)
    }

    return metrics, rows

In [66]:
metrics, rows = evaluate_hors_metrics(results, combined)

for k, v in metrics.items():
    if isinstance(v, float):
        print(k, round(v, 4))
    else:
        print(k, v)

accuracy 0.975
precision 0.9571
recall 0.971
f1 0.964
specificity 0.9771
balanced_accuracy 0.9741
tp 67
fp 3
tn 128
fn 2
total 200


### 6.4. Analyse des faux positifs et des faux négatifs

Les erreurs sont séparées en deux catégories. Les faux positifs correspondent aux articles que le modèle exclut à tort du corpus. Les faux négatifs correspondent aux articles hors-échantillon que le modèle conserve à tort. Cette distinction est importante pour choisir le modèle le plus adapté à la constitution finale du corpus.

In [67]:
false_positives = [r for r in rows if r["predicted"] is True and r["annotated"] is False]
false_negatives = [r for r in rows if r["predicted"] is False and r["annotated"] is True]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 3
False negatives: 2


### 6.5. Export des erreurs pour analyse qualitative

Cette dernière cellule sauvegarde les articles mal classés dans un fichier séparé. Ce fichier permet de revenir au texte des articles problématiques, d’identifier les zones d’ambiguïté du prompt et de comparer qualitativement les erreurs produites par différents modèles.

In [68]:
def save_wrong_hors_comparison(predicted, annotated, output_path):
    n = min(len(predicted), len(annotated))

    wrong = []

    for i in range(n):
        pred = predicted[i]["hors_echantillon"]
        true = annotated[i]["hors_echantillon"]

        if pred != true:
            wrong.append({
                "index": i,
                "id_predicted": predicted[i].get("id"),
                "id_annotated": annotated[i].get("id"),
                "text": annotated[i].get("text", ""),
                "llm_hors_echantillon": pred,
                "annotated_hors_echantillon": true
            })

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(wrong, f, ensure_ascii=False, indent=2)

    return wrong


output_path = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/wrong_hors_comparison_gptmini3.json"

wrong_hors = save_wrong_hors_comparison(
    results,
    combined,
    output_path
)

print("Errors:", len(wrong_hors))
print("File saved:", output_path)
print(wrong_hors[:3])

Errors: 5
File saved: /Users/quentinnippert/Documents/mm_files/TASS_analyse/COE_discours_detection/wrong_hors_comparison_gptmini3.json
[{'index': 106, 'id_predicted': 107, 'id_annotated': 107, 'text': 'Официальный представитель министерства обороны РФ Игорь Конашенков\nМОСКВА, 8 марта. /ТАСС/. Российская сторона с 10:00 мск ввела режим тишины, открыты гуманитарные коридоры из Киева, Чернигова, Сум, Харькова и Мариуполя.\nОб этом заявил официальный представитель Минобороны РФ генерал-майор Игорь Конашенков.\n"В целях безопасной эвакуации мирных граждан из населенных пунктов сегодня с 10:00 московского времени вводится режим тишины и открываются гуманитарные коридоры из Киева, Чернигова, Сум, Харькова и Мариуполя", - сказал он.\nПрезидент России Владимир Путин 24 февраля объявил о проведении специальной военной операции на Украине в ответ на обращение руководителей республик Донбасса о помощи. Он подчеркнул, что в планы Москвы не входит оккупация украинских территорий, целью является дем